# Environment Setup

Run this notebook once before running any experiment notebooks.
It handles all environment setup, authentication, and dataset download.
Safe to run multiple times — skips steps that are already complete.

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

sys.path.insert(0, repo_path)
print(f"Repo ready at: {repo_path}")

## Step 1: Install Dependencies

In [ ]:
!pip install anomalib==2.3.3 einops timm kornia lightning scikit-image huggingface_hub ADEval -q

import anomalib
print(f"Anomalib version: {anomalib.__version__}")
print("All dependencies installed")

## Step 2: Hugging Face Authentication

Real-IAD requires a Hugging Face account and access token.
1. Go to huggingface.co/settings/tokens and create a read token
2. In Colab click the key icon on the left sidebar
3. Add a secret called HF_TOKEN with your token value
4. Request access to Real-IAD/Real-IAD on Hugging Face if not already approved

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print("Hugging Face token configured")

## Step 3: Download Real-IAD Dataset (512px)

Downloads all 30 category zip files and JSON metadata from Hugging Face.
Skips download if files already exist. Takes approximately 10-15 minutes on first run.

In [ ]:
from huggingface_hub import snapshot_download
import os

dataset_dir = '/content/drive/MyDrive/datasets/realiad_512'
zip_dir = os.path.join(dataset_dir, 'realiad_512')
os.makedirs(dataset_dir, exist_ok=True)

# Check if already downloaded
existing_zips = [f for f in os.listdir(zip_dir) 
                 if f.endswith('.zip')] if os.path.exists(zip_dir) else []

if len(existing_zips) >= 30:
    print(f"All 30 category zips already present — skipping download")
else:
    print("Downloading Real-IAD 512px zip files from Hugging Face...")
    snapshot_download(
        repo_id='Real-IAD/Real-IAD',
        repo_type='dataset',
        local_dir=dataset_dir,
        allow_patterns='realiad_512/*',
        token=os.environ['HF_TOKEN']
    )
    print("Category zips downloaded")

In [ ]:
from huggingface_hub import hf_hub_download

json_zip_path = os.path.join(dataset_dir, 'realiad_jsons.zip')

if os.path.exists(json_zip_path):
    print("JSON zip already present — skipping download")
else:
    print("Downloading JSON metadata...")
    hf_hub_download(
        repo_id='Real-IAD/Real-IAD',
        repo_type='dataset',
        filename='realiad_jsons.zip',
        token=os.environ['HF_TOKEN'],
        local_dir=dataset_dir
    )
    print(f"JSON zip downloaded")

## Step 4: Unzip Dataset

In [ ]:
import zipfile

zip_dir = os.path.join(dataset_dir, 'realiad_512')

# Check if already unzipped by looking for category folders
existing_categories = [f for f in os.listdir(dataset_dir) 
                       if os.path.isdir(os.path.join(dataset_dir, f)) 
                       and f not in ['realiad_512', 'realiad_jsons']]

if len(existing_categories) >= 30:
    print(f"All 30 categories already unzipped — skipping")
else:
    zip_files = sorted([f for f in os.listdir(zip_dir) if f.endswith('.zip')])
    print(f"Unzipping {len(zip_files)} category zip files...")
    for zip_file in zip_files:
        zip_path = os.path.join(zip_dir, zip_file)
        category = zip_file.replace('.zip', '')
        if not os.path.exists(os.path.join(dataset_dir, category)):
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(dataset_dir)
            print(f"  Unzipped {category}")
        else:
            print(f"  Skipping {category} — already exists")
    print("All categories unzipped")

In [ ]:
json_zip_path = os.path.join(dataset_dir, 'realiad_jsons.zip')
json_dir = os.path.join(dataset_dir, 'realiad_jsons')

if os.path.exists(json_dir):
    print("JSON metadata already unzipped — skipping")
else:
    print("Unzipping JSON metadata...")
    with zipfile.ZipFile(json_zip_path, 'r') as z:
        z.extractall(dataset_dir)
    print("JSON metadata unzipped")

## Step 5: Verify Setup

In [ ]:
import os

print("=== Setup Verification ===\n")

# Check repo
print(f"Repo: {'✓' if os.path.exists(repo_path) else '✗'} {repo_path}")

# Check categories
categories = [f for f in os.listdir(dataset_dir) 
              if os.path.isdir(os.path.join(dataset_dir, f))
              and f not in ['realiad_512', 'realiad_jsons']]
print(f"Categories: {'✓' if len(categories) >= 30 else '✗'} {len(categories)}/30 found")

# Check JSONs
json_dir = os.path.join(dataset_dir, 'realiad_jsons')
if os.path.exists(json_dir):
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    print(f"JSON files: {'✓' if len(json_files) >= 30 else '✗'} {len(json_files)} found")
else:
    # Check nested structure
    nested_json = os.path.join(dataset_dir, 'realiad_jsons', 'realiad_jsons')
    if os.path.exists(nested_json):
        json_files = [f for f in os.listdir(nested_json) if f.endswith('.json')]
        print(f"JSON files: {'✓' if len(json_files) >= 30 else '✗'} {len(json_files)} found (nested)")
    else:
        print("JSON files: ✗ not found")

print("\nSetup complete — ready to run experiments")